# Lecture 23: System Identification

We have a torque-controlled Panda whose true dynamics differ from the URDF. Goal: recover 28 unknown parameters (armature, damping, link mass, Coulomb friction — one per joint) from a single torque-driven trajectory, using the Cross-Entropy Method (CEM) for optimization.

We'll save the elite mean from each CEM iteration and replay them in viser as a transparent "ghost" alongside the real robot.

In [1]:
import time
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import mujoco
from mujoco import rollout as mj_rollout

import robot_descriptions.panda_mj_description as _panda_mj
import robot_descriptions.panda_description as _panda_urdf
from robot_descriptions.loaders.mujoco import load_robot_description

import viser
from viser.extras import ViserUrdf

## Load the robot

Identified parameters are *multipliers* on URDF defaults (except friction, which is absolute Nm), so the search space stays roughly unit-scale.

In [2]:
_MJCF_PATH = _panda_mj.MJCF_PATH
model_ref = load_robot_description("panda_mj_description")
NJ = 7
NU = model_ref.nu

# actuator index → dof / body index
_dof, _body = [], []
for i in range(NJ):
    jid = model_ref.actuator(i).trnid[0]
    _dof.append(model_ref.jnt_dofadr[jid])
    _body.append(model_ref.jnt_bodyid[jid])

default_armature = np.array([model_ref.dof_armature[d]     for d in _dof])
default_damping  = np.array([model_ref.dof_damping[d]      for d in _dof])
default_mass     = np.array([model_ref.body_mass[b]        for b in _body])
default_friction = np.array([model_ref.dof_frictionloss[d] for d in _dof])

## "True" parameters

In place of real hardware, we pick ground-truth values far from the URDF defaults and simulate them.

In [3]:
TRUE_ARM  = np.array([6.0, 0.5, 4.0, 8.0, 1.0, 5.0, 2.0])
TRUE_DMP  = np.array([3.0, 0.3, 5.0, 0.5, 4.0, 0.8, 3.5])
TRUE_MASS = np.array([1.8, 0.5, 2.5, 0.7, 2.0, 0.6, 1.5])
TRUE_FRIC = np.array([1.5, 0.3, 2.0, 0.5, 1.0, 1.5, 0.3])

N_PARAMS = NJ * 4
pack   = lambda a, d, m, f: np.concatenate([a, d, m, f])
unpack = lambda p: (p[:NJ], p[NJ:2*NJ], p[2*NJ:3*NJ], p[3*NJ:])

TRUE_PARAMS = pack(TRUE_ARM, TRUE_DMP, TRUE_MASS, TRUE_FRIC)
INIT_PARAMS = pack(np.ones(NJ), np.ones(NJ), np.ones(NJ), np.zeros(NJ))

## Sim config and model pool

We pre-allocate a pool of `MjModel`s so each CEM iteration just mutates parameters in place. Observations are subsampled to 50 Hz to mimic encoder rates.

In [4]:
DT      = model_ref.opt.timestep
SUBSTEP = 10
T_SIM   = 4.0
N_STEPS = int(T_SIM / DT)
OBS_IDX = np.arange(0, N_STEPS, SUBSTEP)
N_OBS   = len(OBS_IDX)
POP     = 512
NTHREAD = 128

STATE_SIZE = mujoco.mj_stateSize(model_ref, mujoco.mjtState.mjSTATE_FULLPHYSICS)
QPOS_SL = slice(1, 1 + model_ref.nq)

# neutral joint config (midpoints of ranges)
q0 = np.zeros(NJ)
for i in range(NJ):
    jid = model_ref.actuator(i).trnid[0]
    q0[i] = 0.5 * (model_ref.jnt_range[jid, 0] + model_ref.jnt_range[jid, 1])

print(f"{N_PARAMS} params | POP={POP} | {N_OBS} obs @ {1/(DT*SUBSTEP):.0f} Hz | T={T_SIM}s")

28 params | POP=512 | 200 obs @ 50 Hz | T=4.0s


In [5]:
def apply_params(model, params):
    arm, dmp, mass, fric = unpack(params)
    for i in range(NJ):
        model.dof_armature[_dof[i]]     = arm[i]  * default_armature[i]
        model.dof_damping[_dof[i]]      = dmp[i]  * default_damping[i]
        model.body_mass[_body[i]]       = mass[i] * default_mass[i]
        model.dof_frictionloss[_dof[i]] = fric[i]

def make_model(params):
    """Fresh torque-mode Panda with given params."""
    m = mujoco.MjModel.from_xml_path(_MJCF_PATH)
    for i in range(NJ):
        m.actuator_gainprm[i, 0] = 1.0
        m.actuator_biasprm[i, :] = 0.0
    apply_params(m, params)
    return m

# gravity comp at neutral, using NOMINAL model (deliberately wrong on "real" robot)
_m0 = make_model(INIT_PARAMS)
_d0 = mujoco.MjData(_m0)
for i in range(NJ):
    _d0.qpos[_m0.jnt_qposadr[_m0.actuator(i).trnid[0]]] = q0[i]
mujoco.mj_forward(_m0, _d0)
TAU_GRAV = _d0.qfrc_bias[: _m0.nv].copy()[:NJ]

INIT_STATE = np.zeros(STATE_SIZE)
mujoco.mj_getState(_m0, _d0, INIT_STATE, mujoco.mjtState.mjSTATE_FULLPHYSICS)

print("Allocating model pool...", end=" ", flush=True)
t0 = time.time()
_pool  = [make_model(INIT_PARAMS) for _ in range(POP)]
_tdata = [mujoco.MjData(model_ref) for _ in range(NTHREAD)]
print(f"done ({time.time()-t0:.1f}s)")

Allocating model pool... done (6.1s)


## Excitation trajectories

Sum-of-sinusoids torques on top of gravity comp — standard sysID excitation. We build separate `train` and `test` trajectories with different frequencies.

In [6]:
def make_ctrl(kind, seed=0):
    """Sum-of-sinusoids excitation on top of gravity compensation."""
    rng = np.random.RandomState(seed)
    t = np.arange(N_STEPS) * DT
    ctrl = np.tile(TAU_GRAV, (N_STEPS, 1))
    amp = np.array([15, 15, 10, 10, 5, 4, 2.0])
    if kind == "train":
        freqs_of = lambda j: [0.4 + 0.25*j, 1.0 + 0.2*j, 2.2 - 0.1*j, 3.0 + 0.15*j]
        scale = 1.0
    else:
        freqs_of = lambda j: [0.6 + 0.2*j, 1.5 - 0.1*j, 2.8 + 0.1*j]
        scale = 0.8
    for j in range(NJ):
        freqs = freqs_of(j)
        phases = rng.uniform(0, 2*np.pi, len(freqs))
        for f, phi in zip(freqs, phases):
            ctrl[:, j] += scale * amp[j] * np.sin(2*np.pi*f*t + phi)
    # Pad with zero gripper actuation to match model.nu.
    pad = np.zeros((N_STEPS, NU - NJ))
    return np.concatenate([ctrl, pad], axis=1)

def rollout_batch(models, ctrl):
    """Roll out a batch of models under a shared control trajectory.
    Returns just the 7 arm-joint angles, subsampled to OBS_IDX."""
    nb = len(models)
    states, _ = mj_rollout.rollout(
        models, _tdata,
        np.tile(INIT_STATE, (nb, 1)),
        np.tile(ctrl, (nb, 1, 1)),
        nstep=N_STEPS,
    )
    return states[:, OBS_IDX][:, :, QPOS_SL][:, :, :NJ]

def rollout_one(model, ctrl):
    return rollout_batch([model], ctrl)[0]

## Generate "hardware" data

Roll out the true model, add encoder noise, and check how badly the nominal sim mispredicts.

In [7]:
model_true  = make_model(TRUE_PARAMS)
ctrl_train  = make_ctrl("train", seed=0)
ctrl_test   = make_ctrl("test",  seed=42)

q_hw_train  = rollout_one(model_true, ctrl_train)
q_hw_test   = rollout_one(model_true, ctrl_test)

rng = np.random.RandomState(99)
q_hw_train += rng.randn(*q_hw_train.shape) * 5e-4
q_hw_test  += rng.randn(*q_hw_test.shape)  * 5e-4

q_nom_test = rollout_one(make_model(INIT_PARAMS), ctrl_test)
nom_err = np.degrees(np.max(np.abs(q_nom_test - q_hw_test)))
nom_rms = np.degrees(np.sqrt(np.mean((q_nom_test - q_hw_test) ** 2)))
print(f"Nominal sim: max |Δq| = {nom_err:.1f}°, RMS = {nom_rms:.1f}°")

Nominal sim: max |Δq| = 38.3°, RMS = 14.5°


## CEM

Sample parameters from a Gaussian, roll out the sim, keep the elite samples, refit the Gaussian. We snapshot the elite mean each iteration for later replay.

In [8]:
N_ITERS = 20

def cem(ctrl, q_target, n_iters=N_ITERS, seed=42):
    rng = np.random.RandomState(seed)
    mu = INIT_PARAMS.copy()
    sigma = np.concatenate([
        2.0*np.ones(NJ), 2.0*np.ones(NJ), 0.8*np.ones(NJ), 1.0*np.ones(NJ),
    ])
    n_elite = int(POP * 0.08)
    lo = np.concatenate([0.1*np.ones(3*NJ), np.zeros(NJ)])
    hi = np.concatenate([10 *np.ones(3*NJ), 5*np.ones(NJ)])

    mu_hist, best_hist, cost_hist = [], [], []
    for it in range(n_iters):
        samples = np.clip(rng.randn(POP, N_PARAMS) * sigma + mu, lo, hi)
        for k in range(POP):
            apply_params(_pool[k], samples[k])
        qpos = rollout_batch(_pool, ctrl)
        costs = np.mean((qpos - q_target[None]) ** 2, axis=(1, 2))
        idx = np.argsort(costs)[:n_elite]
        mu    = np.mean(samples[idx], axis=0)
        sigma = np.std (samples[idx], axis=0) + 1e-4

        mu_hist.append(mu.copy())
        best_hist.append(samples[idx[0]].copy())
        cost_hist.append(costs[idx[0]])

        a, d, m, f = unpack(mu)
        print(f"  iter {it:2d} | cost {costs[idx[0]]:.4f} | "
              f"arm:{np.mean(np.abs(a-TRUE_ARM)):.2f} "
              f"dmp:{np.mean(np.abs(d-TRUE_DMP)):.2f} "
              f"mass:{np.mean(np.abs(m-TRUE_MASS)):.2f} "
              f"fric:{np.mean(np.abs(f-TRUE_FRIC)):.2f}")
    return np.array(mu_hist), np.array(best_hist), np.array(cost_hist)

print("CEM: identifying 28 parameters from training trajectory")
t0 = time.time()
mu_hist, best_hist, cost_hist = cem(ctrl_train, q_hw_train)
print(f"Time: {time.time()-t0:.1f}s")
params_id = mu_hist[-1]

CEM: identifying 28 parameters from training trajectory
  iter  0 | cost 0.0106 | arm:2.54 dmp:1.45 mass:0.75 fric:0.55
  iter  1 | cost 0.0071 | arm:2.42 dmp:1.25 mass:0.77 fric:0.50
  iter  2 | cost 0.0051 | arm:2.22 dmp:1.26 mass:0.70 fric:0.56
  iter  3 | cost 0.0034 | arm:2.14 dmp:1.19 mass:0.63 fric:0.51
  iter  4 | cost 0.0027 | arm:1.90 dmp:1.09 mass:0.63 fric:0.50
  iter  5 | cost 0.0019 | arm:1.77 dmp:1.08 mass:0.67 fric:0.47
  iter  6 | cost 0.0016 | arm:1.64 dmp:1.07 mass:0.66 fric:0.43
  iter  7 | cost 0.0014 | arm:1.53 dmp:1.00 mass:0.68 fric:0.46
  iter  8 | cost 0.0009 | arm:1.43 dmp:0.95 mass:0.72 fric:0.46
  iter  9 | cost 0.0007 | arm:1.41 dmp:0.89 mass:0.75 fric:0.45
  iter 10 | cost 0.0004 | arm:1.33 dmp:0.89 mass:0.77 fric:0.44
  iter 11 | cost 0.0004 | arm:1.29 dmp:0.88 mass:0.75 fric:0.43
  iter 12 | cost 0.0003 | arm:1.29 dmp:0.88 mass:0.78 fric:0.40
  iter 13 | cost 0.0003 | arm:1.29 dmp:0.87 mass:0.74 fric:0.38
  iter 14 | cost 0.0003 | arm:1.26 dmp:0.86 mass

## Replay each iteration

Re-roll out every iteration's elite mean at full sim resolution on the **held-out** test trajectory — the ghost should converge to the real robot only if the identified parameters generalize beyond the training excitation.

In [9]:
# Rollout the elite mean from each iteration at full resolution (no subsampling)
# so playback is smooth in viser. We evaluate on the held-out test trajectory.
def rollout_full(params, ctrl):
    m = _pool[0]
    apply_params(m, params)
    states, _ = mj_rollout.rollout(
        [m], [_tdata[0]],
        INIT_STATE[None], ctrl[None], nstep=N_STEPS,
    )
    return states[0, :, QPOS_SL][:, :NJ]  # (N_STEPS, NJ)

# "Real" rollout at full resolution on the held-out test trajectory
q_hw_full = rollout_full(TRUE_PARAMS, ctrl_test)
rng_play = np.random.RandomState(99)
q_hw_full = q_hw_full + rng_play.randn(*q_hw_full.shape) * 5e-4

# Per-iteration simulated rollout at full resolution on the test trajectory
q_sim_hist = np.stack([rollout_full(mu_hist[i], ctrl_test) for i in range(N_ITERS)])
print(f"q_hw_full: {q_hw_full.shape}, q_sim_hist: {q_sim_hist.shape}")

q_hw_full: (2000, 7), q_sim_hist: (20, 2000, 7)


## Viser: real vs simulated ghost

The slider scrubs CEM iterations — the orange ghost should snap onto the real robot as the parameters converge.

In [10]:
URDF_PATH = Path(_panda_urdf.URDF_PATH)

if "server" not in globals():
    server = viser.ViserServer()
else:
    server.scene.reset()
    server.gui.reset()

# Real robot — normal appearance
real_root  = server.scene.add_frame("/real",  show_axes=False)
urdf_real  = ViserUrdf(server, URDF_PATH, root_node_name="/real")

# Ghost — transparent orange, co-located with real
ghost_root = server.scene.add_frame("/ghost", show_axes=False)
urdf_ghost = ViserUrdf(server, URDF_PATH, root_node_name="/ghost",
                       mesh_color_override=(1.0, 0.55, 0.1, 0.45))

# viser's ViserUrdf expects configs in actuated-joint order
joint_names = urdf_real.get_actuated_joint_names()
print("Actuated joints:", joint_names)

╭────── viser (listening *:8080) ───────╮
│             ╷                         │
│   HTTP      │ http://localhost:8080   │
│   Websocket │ ws://localhost:8080     │
│             ╵                         │
╰───────────────────────────────────────╯

Actuated joints: ('panda_joint1', 'panda_joint2', 'panda_joint3', 'panda_joint4', 'panda_joint5', 'panda_joint6', 'panda_joint7', 'panda_finger_joint1')


In [11]:
# For the Panda the first 7 actuated joints line up with our NJ=7 arm joints.
# Pad with zeros if the URDF has extra actuated joints (e.g., fingers).
N_URDF_JOINTS = len(joint_names)

def cfg_from_q(q7):
    cfg = np.zeros(N_URDF_JOINTS)
    cfg[:NJ] = q7
    return cfg

iter_slider = server.gui.add_slider(
    "CEM iteration", min=0, max=N_ITERS - 1, step=1, initial_value=N_ITERS - 1,
)
frame_slider = server.gui.add_slider(
    "Frame", min=0, max=N_STEPS - 1, step=1, initial_value=0,
)
play_button  = server.gui.add_button("Play")
cost_label   = server.gui.add_markdown(f"**Iter cost:** {cost_hist[-1]:.4f}")

def render(it, t):
    urdf_real .update_cfg(cfg_from_q(q_hw_full[t]))
    urdf_ghost.update_cfg(cfg_from_q(q_sim_hist[it, t]))

@iter_slider.on_update
def _(_):
    cost_label.content = f"**Iter cost:** {cost_hist[iter_slider.value]:.4f}"
    render(iter_slider.value, frame_slider.value)

@frame_slider.on_update
def _(_):
    render(iter_slider.value, frame_slider.value)

@play_button.on_click
def _(_):
    # Stride 2 keeps playback close to wall-clock real time.
    for t in range(0, N_STEPS, 2):
        frame_slider.value = t
        render(iter_slider.value, t)
        time.sleep(DT * 2)

render(iter_slider.value, 0)
print("Viser ready. Drag the 'CEM iteration' slider to watch the ghost converge to the real robot.")

Viser ready. Drag the 'CEM iteration' slider to watch the ghost converge to the real robot.


## Held-out evaluation

The real test: do identified parameters predict the response to a *different* excitation?

In [ ]:
model_id = make_model(params_id)
q_id_test = rollout_one(model_id, ctrl_test)
id_err = np.degrees(np.max(np.abs(q_id_test - q_hw_test)))
id_rms = np.degrees(np.sqrt(np.mean((q_id_test - q_hw_test) ** 2)))

print(f"{'Nominal':<12s} max |Δq|: {nom_err:6.1f}°   RMS: {nom_rms:.1f}°")
print(f"{'Identified':<12s} max |Δq|: {id_err :6.1f}°   RMS: {id_rms :.1f}°")
print(f"Improvement: {nom_rms/max(id_rms, 0.01):.1f}× RMS reduction")

arm_id, dmp_id, mass_id, fric_id = unpack(params_id)
for name, true, ident in [
    ("Armature", TRUE_ARM, arm_id),
    ("Damping",  TRUE_DMP, dmp_id),
    ("Mass",     TRUE_MASS, mass_id),
    ("Friction", TRUE_FRIC, fric_id),
]:
    print(f"\n{name}:")
    print(f"  True: [{', '.join(f'{v:.2f}' for v in true)}]")
    print(f"  ID:   [{', '.join(f'{v:.2f}' for v in ident)}]")
    print(f"  Mean |error|: {np.mean(np.abs(ident - true)):.3f}")

Nominal      max |Δq|:   38.3°   RMS: 14.5°
Identified   max |Δq|:    8.2°   RMS: 2.1°
Improvement: 6.8× RMS reduction

Armature:
  True: [6.00, 0.50, 4.00, 8.00, 1.00, 5.00, 2.00]
  ID:   [2.34, 0.31, 3.34, 5.24, 1.95, 5.03, 2.10]
  Mean |error|: 1.194

Damping:
  True: [3.00, 0.30, 5.00, 0.50, 4.00, 0.80, 3.50]
  ID:   [4.82, 0.32, 3.00, 0.86, 3.54, 1.60, 3.21]
  Mean |error|: 0.822

Mass:
  True: [1.80, 0.50, 2.50, 0.70, 2.00, 0.60, 1.50]
  ID:   [1.30, 1.91, 1.90, 0.12, 0.73, 1.27, 1.18]
  Mean |error|: 0.765

Friction:
  True: [1.50, 0.30, 2.00, 0.50, 1.00, 1.50, 0.30]
  ID:   [1.30, 0.21, 2.15, 0.17, 0.63, 0.37, 0.45]
  Mean |error|: 0.345


(viser) Connection opened (0, 1 total), 336 persistent messages

(viser) Connection closed (0, 0 total)

(viser) Connection opened (1, 1 total), 337 persistent messages